# Lab 02 - End-to-End Machine Learning Project

Notebook này giữ đầy đủ flow của bài California Housing, nhưng phần code dùng lại được đã được refactor vào package `lab02` trong `src/lab02`.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from packaging import version
from pandas.plotting import scatter_matrix
from scipy import stats
from scipy.stats import binom, expon, loguniform, randint
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

from lab02.config import IMAGES_DIR, MODELS_DIR, PROJECT_ROOT, RANDOM_STATE
from lab02.data import (
    add_income_category,
    is_id_in_test_set,
    load_housing_data,
    shuffle_and_split_data,
    split_data_with_id_hash,
    split_features_labels,
    stratified_train_test_split,
)
from lab02.evaluation import evaluate_rmse, rmse_confidence_interval
from lab02.modeling import (
    build_linear_regression_pipeline,
    build_random_forest_pipeline,
    build_tree_regression_pipeline,
    cross_validate_rmse,
    load_model,
    random_search_random_forest,
    save_model,
)
from lab02.preprocessing import build_preprocessing_pipeline, column_ratio, ratio_pipeline
from lab02.transformers import ClusterSimilarity, FeatureFromRegressor, StandardScalerClone
from lab02.visualization import configure_matplotlib, save_fig

assert sys.version_info >= (3, 9)
assert version.parse(sklearn.__version__) >= version.parse("1.5")

np.random.seed(RANDOM_STATE)
configure_matplotlib()
PROJECT_ROOT

## 1. Get The Data

In [ ]:
housing = load_housing_data()
housing.head()

In [ ]:
housing.info()

In [ ]:
housing["ocean_proximity"].value_counts()

In [ ]:
housing.describe()

In [ ]:
housing.hist(bins=50, figsize=(12, 8))
save_fig("attribute_histogram_plots")
plt.show()

## 2. Create A Test Set

In [ ]:
train_set, test_set = shuffle_and_split_data(housing, test_ratio=0.2, random_state=RANDOM_STATE)
len(train_set), len(test_set)

In [ ]:
housing_with_id = housing.reset_index()
train_set_hash, test_set_hash = split_data_with_id_hash(housing_with_id, 0.2, "index")
len(train_set_hash), len(test_set_hash)

In [ ]:
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set_hash, test_set_hash = split_data_with_id_hash(housing_with_id, 0.2, "id")
len(train_set_hash), len(test_set_hash)

In [ ]:
train_set_random, test_set_random = train_test_split(housing, test_size=0.2, random_state=RANDOM_STATE)
test_set_random["total_bedrooms"].isnull().sum()

In [ ]:
sample_size = 1000
ratio_female = 0.511
proba_too_small = binom(sample_size, ratio_female).cdf(485 - 1)
proba_too_large = 1 - binom(sample_size, ratio_female).cdf(535)
proba_too_small + proba_too_large

In [ ]:
housing_with_income_cat = add_income_category(housing)
housing_with_income_cat["income_cat"].value_counts().sort_index().plot.bar(rot=0, grid=True)
plt.xlabel("Income category")
plt.ylabel("Number of districts")
save_fig("housing_income_cat_bar_plot")
plt.show()

In [ ]:
def income_cat_proportions(data):
    return data["income_cat"].value_counts() / len(data)

strat_train_set, strat_test_set = stratified_train_test_split(housing)
strat_test_with_income_cat = add_income_category(strat_test_set)
random_test_with_income_cat = add_income_category(test_set_random)

compare_props = pd.DataFrame({
    "Overall %": income_cat_proportions(housing_with_income_cat),
    "Stratified %": income_cat_proportions(strat_test_with_income_cat),
    "Random %": income_cat_proportions(random_test_with_income_cat),
}).sort_index()
compare_props.index.name = "Income Category"
compare_props["Strat. Error %"] = compare_props["Stratified %"] / compare_props["Overall %"] - 1
compare_props["Rand. Error %"] = compare_props["Random %"] / compare_props["Overall %"] - 1
(compare_props * 100).round(2)

## 3. Discover And Visualize The Data

In [ ]:
housing_explore = strat_train_set.copy()
housing_explore.plot(kind="scatter", x="longitude", y="latitude", grid=True)
save_fig("bad_visualization_plot")
plt.show()

In [ ]:
housing_explore.plot(kind="scatter", x="longitude", y="latitude", grid=True, alpha=0.2)
save_fig("better_visualization_plot")
plt.show()

In [ ]:
housing_explore.plot(
    kind="scatter",
    x="longitude",
    y="latitude",
    grid=True,
    s=housing_explore["population"] / 100,
    label="population",
    c="median_house_value",
    cmap="jet",
    colorbar=True,
    legend=True,
    sharex=False,
    figsize=(10, 7),
)
save_fig("housing_prices_scatterplot")
plt.show()

In [ ]:
filename = "california.png"
california_path = IMAGES_DIR / filename
if not california_path.is_file():
    import urllib.request
    url = "https://github.com/ageron/handson-ml3/raw/main/images/end_to_end_project/california.png"
    urllib.request.urlretrieve(url, california_path)

housing_renamed = housing_explore.rename(columns={
    "latitude": "Latitude",
    "longitude": "Longitude",
    "population": "Population",
    "median_house_value": "Median house value (USD)",
})
housing_renamed.plot(
    kind="scatter",
    x="Longitude",
    y="Latitude",
    s=housing_renamed["Population"] / 100,
    label="Population",
    c="Median house value (USD)",
    cmap="jet",
    colorbar=True,
    legend=True,
    sharex=False,
    figsize=(10, 7),
)
california_img = plt.imread(california_path)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)
save_fig("california_housing_prices_plot")
plt.show()

## 4. Look For Correlations

In [ ]:
corr_matrix = housing_explore.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
attributes = ["median_house_value", "median_income", "total_rooms", "housing_median_age"]
scatter_matrix(housing_explore[attributes], figsize=(12, 8))
save_fig("scatter_matrix_plot")
plt.show()

In [ ]:
housing_explore.plot(
    kind="scatter",
    x="median_income",
    y="median_house_value",
    alpha=0.1,
    grid=True,
)
save_fig("income_vs_house_value_scatterplot")
plt.show()

## 5. Experiment With Attribute Combinations

In [ ]:
housing_explore["rooms_per_house"] = housing_explore["total_rooms"] / housing_explore["households"]
housing_explore["bedrooms_ratio"] = housing_explore["total_bedrooms"] / housing_explore["total_rooms"]
housing_explore["people_per_house"] = housing_explore["population"] / housing_explore["households"]

corr_matrix = housing_explore.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

## 6. Prepare The Data For Machine Learning

In [ ]:
housing_train, housing_labels = split_features_labels(strat_train_set)
housing_test, housing_test_labels = split_features_labels(strat_test_set)
housing_train.head()

In [ ]:
null_rows_idx = housing_train.isnull().any(axis=1)
housing_train.loc[null_rows_idx].head()

In [ ]:
housing_option1 = housing_train.copy()
housing_option1.dropna(subset=["total_bedrooms"], inplace=True)
housing_option1.loc[null_rows_idx].head()

In [ ]:
housing_option2 = housing_train.copy()
housing_option2.drop("total_bedrooms", axis=1, inplace=True)
housing_option2.loc[null_rows_idx].head()

In [ ]:
housing_option3 = housing_train.copy()
median = housing_train["total_bedrooms"].median()
housing_option3["total_bedrooms"] = housing_option3["total_bedrooms"].fillna(median)
housing_option3.loc[null_rows_idx].head()

In [ ]:
imputer = SimpleImputer(strategy="median")
housing_num = housing_train.select_dtypes(include=[np.number])
imputer.fit(housing_num)
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns, index=housing_num.index)
housing_tr.loc[null_rows_idx].head()

In [ ]:
isolation_forest = IsolationForest(random_state=RANDOM_STATE)
outlier_pred = isolation_forest.fit_predict(X)
pd.Series(outlier_pred).value_counts()

### Categorical Attributes

In [ ]:
housing_cat = housing_train[["ocean_proximity"]]
housing_cat.head(8)

In [ ]:
ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:8], ordinal_encoder.categories_

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)
pd.DataFrame(
    housing_cat_1hot[:8],
    columns=cat_encoder.get_feature_names_out(),
    index=housing_cat.index[:8],
)

### Feature Scaling And Custom Transformers

In [ ]:
min_max_scaler = MinMaxScaler(feature_range=(-1, 1))
housing_num_min_max_scaled = min_max_scaler.fit_transform(housing_num)

std_scaler = StandardScaler()
housing_num_std_scaled = std_scaler.fit_transform(housing_num)
housing_num_min_max_scaled[:3], housing_num_std_scaled[:3]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
housing_train["population"].hist(ax=axs[0], bins=50)
housing_train["population"].apply(np.log).hist(ax=axs[1], bins=50)
axs[0].set_xlabel("Population")
axs[1].set_xlabel("Log of population")
axs[0].set_ylabel("Number of districts")
save_fig("long_tail_plot")
plt.show()

In [ ]:
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=RANDOM_STATE)
similarities = cluster_simil.fit_transform(housing_train[["latitude", "longitude"]])
similarities[:3].round(2)

In [ ]:
preprocessing = build_preprocessing_pipeline()
housing_prepared = preprocessing.fit_transform(housing_train)
housing_prepared.shape

In [ ]:
preprocessing.get_feature_names_out()[:20]

## 7. Select And Train Models

In [ ]:
lin_reg = build_linear_regression_pipeline()
lin_reg.fit(housing_train, housing_labels)

sample_predictions = lin_reg.predict(housing_train.iloc[:5])
pd.DataFrame({"prediction": sample_predictions, "label": housing_labels.iloc[:5].values})

In [ ]:
lin_rmse = evaluate_rmse(lin_reg, housing_train, housing_labels)
lin_rmse

In [ ]:
tree_reg = build_tree_regression_pipeline()
tree_reg.fit(housing_train, housing_labels)
tree_rmse = evaluate_rmse(tree_reg, housing_train, housing_labels)
tree_rmse

In [ ]:
tree_rmses = cross_validate_rmse(tree_reg, housing_train, housing_labels, cv=10)
pd.Series(tree_rmses).describe()

In [ ]:
forest_reg = build_random_forest_pipeline()
forest_rmses = cross_validate_rmse(forest_reg, housing_train, housing_labels, cv=10)
pd.Series(forest_rmses).describe()

## 8. Fine-Tune The Model

In [ ]:
full_pipeline = build_random_forest_pipeline()
param_grid = [
    {"preprocessing__geo__n_clusters": [5, 8, 10], "random_forest__max_features": [4, 6, 8]},
    {"preprocessing__geo__n_clusters": [10, 15], "random_forest__max_features": [6, 8, 10]},
]

grid_search = GridSearchCV(
    full_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
)
grid_search.fit(housing_train, housing_labels)
-grid_search.best_score_, grid_search.best_params_

In [ ]:
rnd_search = random_search_random_forest(
    housing_train,
    housing_labels,
    n_iter=10,
    cv=3,
)
-rnd_search.best_score_, rnd_search.best_params_

In [ ]:
cv_res = pd.DataFrame(rnd_search.cv_results_)
cv_res.sort_values(by="mean_test_score", ascending=False)[[
    "mean_test_score",
    "param_preprocessing__geo__n_clusters",
    "param_random_forest__max_features",
]].head()

In [ ]:
final_model = rnd_search.best_estimator_
feature_importances = final_model["random_forest"].feature_importances_
sorted(
    zip(feature_importances, final_model["preprocessing"].get_feature_names_out()),
    reverse=True,
)[:20]

## 9. Evaluate On The Test Set

In [ ]:
final_predictions = final_model.predict(housing_test)
final_rmse = root_mean_squared_error(housing_test_labels, final_predictions)
final_rmse

In [ ]:
rmse_lower, rmse_upper = rmse_confidence_interval(final_predictions, housing_test_labels)
rmse_lower, rmse_upper

In [ ]:
model_path = save_model(final_model, "california_housing_model.pkl")
model_path

In [ ]:
final_model_reloaded = load_model(model_path)
new_data = housing_train.iloc[:5]
final_model_reloaded.predict(new_data)

## 10. Exercises / Optional Experiments

Các cell dưới đây giữ lại hướng bài tập chính. Một vài cell có thể chạy lâu hơn vì dùng cross-validation hoặc randomized search.

In [ ]:
svr_param_grid = [
    {"svr__kernel": ["linear"], "svr__C": [10.0, 30.0, 100.0, 300.0, 1000.0, 3000.0, 10000.0, 30000.0]},
    {"svr__kernel": ["rbf"], "svr__C": [1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1000.0], "svr__gamma": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]},
]

svr_pipeline = Pipeline([("preprocessing", build_preprocessing_pipeline()), ("svr", SVR())])
svr_grid_search = GridSearchCV(
    svr_pipeline,
    svr_param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
)
svr_grid_search.fit(housing_train.iloc[:5000], housing_labels.iloc[:5000])
-svr_grid_search.best_score_, svr_grid_search.best_params_

In [ ]:
svr_param_distribs = {
    "svr__kernel": ["linear", "rbf"],
    "svr__C": loguniform(20, 200_000),
    "svr__gamma": expon(scale=1.0),
}

svr_rnd_search = RandomizedSearchCV(
    svr_pipeline,
    param_distributions=svr_param_distribs,
    n_iter=50,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
)
svr_rnd_search.fit(housing_train.iloc[:5000], housing_labels.iloc[:5000])
-svr_rnd_search.best_score_, svr_rnd_search.best_params_

In [ ]:
selector_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline()),
    ("selector", SelectFromModel(RandomForestRegressor(random_state=RANDOM_STATE), threshold=0.005)),
    ("svr", SVR(
        C=svr_rnd_search.best_params_["svr__C"],
        gamma=svr_rnd_search.best_params_["svr__gamma"],
        kernel=svr_rnd_search.best_params_["svr__kernel"],
    )),
])

selector_rmses = -cross_val_score(
    selector_pipeline,
    housing_train.iloc[:5000],
    housing_labels.iloc[:5000],
    scoring="neg_root_mean_squared_error",
    cv=3,
)
pd.Series(selector_rmses).describe()

In [ ]:
knn_reg = KNeighborsRegressor(n_neighbors=3, weights="distance")
knn_transformer = FeatureFromRegressor(knn_reg)
geo_features = housing_train[["latitude", "longitude"]]
knn_transformer.fit_transform(geo_features, housing_labels)[:5]

In [ ]:
transformers = [(name, clone(transformer), columns) for name, transformer, columns in build_preprocessing_pipeline().transformers]
geo_index = [name for name, _, _ in transformers].index("geo")
transformers[geo_index] = ("geo", knn_transformer, ["latitude", "longitude"])
new_geo_preprocessing = ColumnTransformer(transformers)

new_geo_pipeline = Pipeline([
    ("preprocessing", new_geo_preprocessing),
    ("svr", SVR(
        C=svr_rnd_search.best_params_["svr__C"],
        gamma=svr_rnd_search.best_params_["svr__gamma"],
        kernel=svr_rnd_search.best_params_["svr__kernel"],
    )),
])

new_pipe_rmses = -cross_val_score(
    new_geo_pipeline,
    housing_train.iloc[:5000],
    housing_labels.iloc[:5000],
    scoring="neg_root_mean_squared_error",
    cv=3,
)
pd.Series(new_pipe_rmses).describe()

In [ ]:
X_scaler_test = np.random.rand(1000, 3)
scaler = StandardScalerClone()
X_scaled = scaler.fit_transform(X_scaler_test)
X_back = scaler.inverse_transform(X_scaled)

assert np.allclose(X_scaled, (X_scaler_test - X_scaler_test.mean(axis=0)) / X_scaler_test.std(axis=0))
assert np.allclose(X_scaler_test, X_back)
assert np.all(scaler.get_feature_names_out() == ["x0", "x1", "x2"])
"StandardScalerClone works"